# Chlamydia trachomatis host-response reanalysis (GEO) with a trained-immunity comparison. Version 2

Run top to bottom in Google Colab (CPU is enough). Python only (PyDESeq2, gseapy).

**Dataset facts confirmed from the files you downloaded**

| Series | What it actually contains | Counts used |
|---|---|---|
| GSE228774 | HVE (vaginal epithelial) only: Mock R1-3 and L2wt R1-3, 6 samples | CLC "Total gene reads" |
| GSE198272 | HCEC (endocervical) : Mock R1-3, L2wt R1-3, plus L2 d480 mutant R1-3 (mutant not used here) | CLC "Total gene reads" |
| GSE180782 | End1 cells: Uninfected R1-3, serovar L2 R1-3, serovar B R1-3, 9 samples | CLC "Total gene reads" |
| GSE158814 | HeLa serovar E: files per group with a "Read Count" column, mixed RNA types | Read Count, mRNA rows only |
| GSE242947 | Macrophages: Untrained, Zymo, ETSU (beta-glucan), Depleted zymosan at 0, 1, 3 h, two replicates | featureCounts table |

The HVE versus HCEC comparison therefore comes from two different GEO series, not one 12-sample series. Infection effects are estimated within each series, so batch does not confound them, but the cell-type comparison is across series.

No result in this notebook is pre-written.

## 0. Setup

In [ ]:
!pip -q install pydeseq2 gseapy statsmodels scikit-learn seaborn requests openpyxl

In [ ]:
import os, re, io, gzip, glob, json, hashlib, warnings, itertools, zipfile, tarfile, importlib.metadata as im
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
import gseapy as gp
warnings.filterwarnings("ignore")

BASE = "/content/ct_reanalysis"
DATA, OUT = f"{BASE}/data", f"{BASE}/results"
FIG = f"{OUT}/figures"
for d in (DATA, OUT, FIG): os.makedirs(d, exist_ok=True)

ACC = {"epi1": "GSE228774+GSE198272", "epi2": "GSE180782", "hela": "GSE158814", "mac": "GSE242947"}
SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9})

for p in ("pydeseq2", "gseapy", "pandas", "numpy", "scipy", "statsmodels", "scikit-learn"):
    try: print(p, im.version(p))
    except Exception: pass

## 1. Download helpers (files are reused if already present)

In [ ]:
FILE_LOG = {}

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def list_suppl(acc):
    stub = acc[:-3] + "nnn"
    url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{acc}/suppl/"
    html = requests.get(url, timeout=60).text
    files = sorted(set(re.findall(r'href="([^"?/][^"]*)"', html)))
    return url, [f for f in files if ":" not in f and f != "filelist.txt"]

def dl_all(acc):
    url, files = list_suppl(acc)
    out = []
    for f in files:
        p = f"{DATA}/{acc}/{f}"; os.makedirs(os.path.dirname(p), exist_ok=True)
        if not os.path.exists(p):
            with requests.get(url + f, stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(p, "wb") as fh:
                    for ch in r.iter_content(1 << 20): fh.write(ch)
        FILE_LOG[f"{acc}/{f}"] = md5(p)
        out.append(p)
    return out

def ensure_extracted(tar_path):
    d = tar_path[:-4]
    if not os.path.isdir(d) or not os.listdir(d):
        os.makedirs(d, exist_ok=True)
        with tarfile.open(tar_path) as t: t.extractall(d)
    return d

ENS2SYM = {}

def load_clc_dir(dirpath, pattern):
    # Per-sample CLC xlsx files. Uses "Total gene reads" (raw read counts), summed by gene symbol.
    cols, rows = {}, []
    for f in sorted(os.listdir(dirpath)):
        mt = re.match(pattern, f)
        if not (f.endswith(".xlsx") and mt): continue
        gsm, grp, rep = mt.group(1), mt.group(2), int(mt.group(3))
        d = pd.read_excel(os.path.join(dirpath, f), sheet_name=0, usecols=["Name", "ENSEMBL", "Total gene reads"])
        d["Name"] = d["Name"].astype(str).str.upper()
        ENS2SYM.update(dict(zip(d["ENSEMBL"].astype(str), d["Name"])))
        cols[gsm] = d.groupby("Name")["Total gene reads"].sum()
        rows.append(dict(gsm=gsm, group=grp, rep=rep, file=f))
    cnt = pd.DataFrame(cols).fillna(0)
    return cnt, pd.DataFrame(rows).set_index("gsm")

## 2. Load all datasets into one structure

`COUNTS[key]` is a genes by samples table of raw counts (gene symbols, upper case). `M[key]` is the sample table with the groups used for statistics. Read the printed tables before continuing.

In [ ]:
for acc in ["GSE228774", "GSE198272", "GSE180782", "GSE158814", "GSE242947"]:
    dl_all(acc)

COUNTS, M = {}, {}

# GSE228774 (HVE) and GSE198272 (HCEC), Mock and wild-type L2 only
c1, m1 = load_clc_dir(ensure_extracted(f"{DATA}/GSE228774/GSE228774_RAW.tar"), r"^(GSM\d+)_HVE_(Mock|L2wt)_R(\d)")
m1["cell"] = "HVE"
c0, m0 = load_clc_dir(ensure_extracted(f"{DATA}/GSE198272/GSE198272_RAW.tar"), r"^(GSM\d+)_\d+_(Mock|L2wt)_R(\d)_GE")
m0["cell"] = "HCEC"
m_epi = pd.concat([m1, m0]); m_epi["cond"] = m_epi["group"].map({"Mock": "mock", "L2wt": "infected"})
COUNTS["epi1"] = pd.concat([c1, c0], axis=1, join="inner")
M["epi1"] = m_epi

# GSE180782 End1 cells: uninfected, L2, B
c2, m2 = load_clc_dir(ensure_extracted(f"{DATA}/GSE180782/GSE180782_RAW.tar"), r"^(GSM\d+)_\d+_(Mock|CtL2|CtB)_R(\d)")
m2["cond"] = m2["group"].map({"Mock": "mock", "CtL2": "L2", "CtB": "B"})
COUNTS["epi2"], M["epi2"] = c2, m2

# GSE158814 HeLa serovar E
HELA_TYPE_REGEX = r"mRNA|protein|coding"
frames, ann = [], {}
for f in sorted(glob.glob(f"{DATA}/GSE158814/*.txt.gz")):
    d = pd.read_csv(f, sep="\t").drop_duplicates("Gene ID")
    ann.update({g: (s, t) for g, s, t in zip(d["Gene ID"], d["Gene Symbol"], d["Type"])})
    rc = d.set_index("Gene ID")[[c for c in d.columns if c.endswith("Read Count")]]
    rc.columns = [c.replace(" Read Count", "") for c in rc.columns]
    frames.append(rc)
hc = pd.concat(frames, axis=1, join="outer").fillna(0)
ha = pd.DataFrame(ann, index=["symbol", "type"]).T
print("GSE158814 RNA types in the files:"); print(ha["type"].value_counts().to_string())
keep = ha.index[ha["type"].str.contains(HELA_TYPE_REGEX, case=False, regex=True)]
if len(keep) == 0: raise ValueError("No mRNA-like type found. Set HELA_TYPE_REGEX from the printed list.")
hc = hc.loc[hc.index.intersection(keep)]
hc.index = ha.loc[hc.index, "symbol"].astype(str).str.upper().values
COUNTS["hela"] = hc.groupby(level=0).sum()
rows = []
for c in COUNTS["hela"].columns:
    mt = re.match(r"^p1_(RB|EB|AB|NC)_(C?)(\d)$", c)
    if not mt: raise ValueError(f"Unrecognised HeLa column {c}")
    g, is_c = mt.group(1), mt.group(2) == "C"
    stage = {"RB": "20h", "EB": "44h", "AB": "44h_IFNg", "NC": "ctrl"}[g]
    cond = "mock" if (g == "NC" or is_c) else "infected"
    rows.append(dict(sample=c, stage=stage, cond=cond, rep=int(mt.group(3))))
M["hela"] = pd.DataFrame(rows).set_index("sample")

# GSE242947 macrophages
md_ = pd.read_csv(f"{DATA}/GSE242947/GSE242947_Cheng_RNAseqCounts.csv.gz", index_col=0)
samp = [c for c in md_.columns if re.match(r"^(Untrained|Zymo|ETSU|Depleted)\.\d+h\.R\d+$", c)]
mc = md_.set_index("Geneid")[samp]
mc.index = mc.index.astype(str).str.replace(r"\.\d+$", "", regex=True)
mapped = mc.index.map(ENS2SYM)
print(f"GSE242947: {pd.Series(mapped).notna().mean()*100:.1f}% of Ensembl IDs mapped to symbols")
mc = mc[pd.Series(mapped).notna().values]; mc.index = [s for s in mapped if isinstance(s, str)]
COUNTS["mac"] = mc.groupby(level=0).sum()
LIG = {"Untrained": "untrained", "Zymo": "zymosan", "ETSU": "beta_glucan", "Depleted": "depleted_zymosan"}
rows = []
for c in COUNTS["mac"].columns:
    lg, tm, rp = c.split(".")
    rows.append(dict(sample=c, ligand=LIG[lg], time=tm, donor=rp))
M["mac"] = pd.DataFrame(rows).set_index("sample")

for k in M:
    print("=" * 80); print(ACC[k], "| genes x samples:", COUNTS[k].shape)
    gc = [c for c in ("cell", "cond", "stage", "ligand", "time") if c in M[k]]
    print(M[k].groupby(gc).size().to_string())
    nonint = bool((COUNTS[k].values % 1 != 0).any())
    print("non-integer values present (will be rounded):", nonint)

Check the tables above. Expected: GSE228774 HVE 3 mock and 3 infected, GSE198272 HCEC 3 and 3, GSE180782 3 each for mock, L2, B, GSE158814 two samples per group, GSE242947 two samples per ligand and time point.

Two assumptions to confirm in the source papers (state them in Methods):
1. In GSE158814 the single uninfected group "NC" is used as the control for both 20 hpi and 44 hpi. The IFN-gamma arm uses its own control "AB_C".
2. In GSE242947 replicate R1 and R2 are different donors, so donor is used as a paired covariate. Set `PAIRED_MAC = False` below if that is wrong.

## 3. Quality control

In [ ]:
def log_cpm(cnt):
    return np.log2(cnt.div(cnt.sum(0), axis=1) * 1e6 + 1)

def qc(key, color_cols, shape_col=None):
    meta, cnt = M[key], COUNTS[key][M[key].index]
    X = log_cpm(cnt[(cnt >= 10).sum(1) >= 2])
    top = X.var(1).sort_values(ascending=False).index[:min(2000, len(X))]
    pca = PCA(n_components=2, random_state=SEED).fit(X.loc[top].T)
    pc = pca.transform(X.loc[top].T)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    hue = meta[color_cols].astype(str).agg(" / ".join, axis=1)
    sns.scatterplot(x=pc[:, 0], y=pc[:, 1], hue=hue.values,
                    style=meta[shape_col].values if shape_col else None, s=70, ax=ax[0])
    ax[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax[0].set_title(f"{ACC[key]} PCA"); ax[0].legend(fontsize=6, bbox_to_anchor=(1, 1))
    sns.heatmap(X.corr(method="spearman"), ax=ax[1], cmap="viridis", cbar_kws={"label": "Spearman"})
    ax[1].set_title("Sample correlation")
    plt.tight_layout(); plt.savefig(f"{FIG}/QC_{ACC[key]}.png"); plt.show()
    print("Library sizes (millions of counted reads):"); print((cnt.sum(0) / 1e6).round(2).to_string())

qc("epi1", ["cell", "cond"])
qc("epi2", ["cond"])
qc("hela", ["stage", "cond"])
qc("mac", ["ligand", "time"], "donor")

Look at the PCA plots. Exclude a sample only with a documented technical reason, add its id to `EXCLUDE`, and re-run from here.

In [ ]:
EXCLUDE = []   # e.g. ["GSM7136157"]; document the reason in the manuscript
for k in M: M[k] = M[k].drop(index=[g for g in EXCLUDE if g in M[k].index])
PAIRED_MAC = True

## 4. Differential expression (PyDESeq2, Benjamini-Hochberg)

Each contrast is fitted only on the samples of the compared groups. With two to three replicates per group, treat all results as exploratory-to-moderate evidence and report effect sizes with adjusted p-values. GSE158814 and GSE242947 have n = 2 per group, so they are not used for discovery.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

def run_de(key, mask, factor, test, ref, label, covariate=None, min_count=10, min_samples=2):
    meta = M[key].loc[mask].copy()
    cnt = COUNTS[key][meta.index].T
    cnt = cnt.loc[:, (cnt >= min_count).sum(0) >= min_samples].round().astype(int)
    n_t, n_r = int((meta[factor] == test).sum()), int((meta[factor] == ref).sum())
    if n_t < 2 or n_r < 2:
        print(f"SKIP {label}: need >=2 per group, have {test}={n_t}, {ref}={n_r}"); return None
    facs = ([covariate] if covariate else []) + [factor]
    md_ = meta[facs].copy()
    try:
        dds = DeseqDataSet(counts=cnt, metadata=md_, design="~" + "+".join(facs), quiet=True)
    except TypeError:
        dds = DeseqDataSet(counts=cnt, metadata=md_, design_factors=facs, quiet=True)
    dds.deseq2()
    try:
        ds = DeseqStats(dds, contrast=[factor, test, ref], quiet=True)
    except TypeError:
        ds = DeseqStats(dds, contrast=[factor, test, ref])
    ds.summary()
    r = ds.results_df.copy()
    r["padj_BH"] = np.nan
    ok = r["pvalue"].notna()
    r.loc[ok, "padj_BH"] = multipletests(r.loc[ok, "pvalue"], method="fdr_bh")[1]
    r["n_test"], r["n_ref"], r["contrast"] = n_t, n_r, label
    r.to_csv(f"{OUT}/DE_{label}.csv")
    print(f"{label}: genes tested={len(r)}, padj<0.05={int((r.padj_BH < 0.05).sum())}, "
          f"up={int(((r.padj_BH < 0.05) & (r.log2FoldChange > 0)).sum())}, "
          f"down={int(((r.padj_BH < 0.05) & (r.log2FoldChange < 0)).sum())}")
    return r

DE = {}
ACC_OF = {"HVE": "GSE228774", "HCEC": "GSE198272"}

m = M["epi1"]
for cell in sorted(m["cell"].unique()):
    lab = f"{ACC_OF[cell]}_{cell}_inf_vs_mock"
    DE[lab] = run_de("epi1", m["cell"] == cell, "cond", "infected", "mock", lab)

m = M["epi2"]
for sv in ("L2", "B"):
    lab = f"GSE180782_{sv}_vs_mock"
    DE[lab] = run_de("epi2", m["cond"].isin([sv, "mock"]), "cond", sv, "mock", lab)

m = M["hela"]
for st, grp in (("20h", ["20h", "ctrl"]), ("44h", ["44h", "ctrl"]), ("44h_IFNg", ["44h_IFNg"])):
    lab = f"GSE158814_{st}_inf_vs_mock"
    DE[lab] = run_de("hela", m["stage"].isin(grp), "cond", "infected", "mock", lab)

m = M["mac"]
for tm in sorted(m["time"].unique()):
    for lg in ("zymosan", "beta_glucan", "depleted_zymosan"):
        lab = f"GSE242947_{lg}_vs_untrained_{tm}"
        DE[lab] = run_de("mac", (m["time"] == tm) & m["ligand"].isin([lg, "untrained"]), "ligand", lg, "untrained", lab,
                         covariate="donor" if PAIRED_MAC else None)
DE = {k: v for k, v in DE.items() if v is not None}

### 4b. Cell-type difference in the infection response (approximate)

Compares the independent HVE and HCEC infection log2 fold changes with their standard errors (z-test on the difference of two independent estimates). This is not a full interaction model and the two cell types come from different series. Say so in Methods.

In [ ]:
ks = [k for k in DE if k.endswith("_HVE_inf_vs_mock") or k.endswith("_HCEC_inf_vs_mock")]
if len(ks) == 2:
    a, b = DE[ks[0]], DE[ks[1]]
    j = a[["log2FoldChange", "lfcSE"]].join(b[["log2FoldChange", "lfcSE"]], lsuffix="_A", rsuffix="_B", how="inner").dropna()
    j["diff_log2FC"] = j.log2FoldChange_A - j.log2FoldChange_B
    j["z"] = j.diff_log2FC / np.sqrt(j.lfcSE_A**2 + j.lfcSE_B**2)
    j["pvalue"] = 2 * stats.norm.sf(np.abs(j.z))
    j["padj_BH"] = multipletests(j.pvalue, method="fdr_bh")[1]
    j.sort_values("pvalue").to_csv(f"{OUT}/celltype_difference_HVE_vs_HCEC.csv")
    print("A =", ks[0], "| B =", ks[1], "| genes:", len(j), "| padj<0.05:", int((j.padj_BH < 0.05).sum()))
else:
    print("Skipped. Found:", ks)

## 5. Volcano plots

In [ ]:
def volcano(r, title, fname, lfc=1.0, alpha=0.05, label_n=12):
    d = r.dropna(subset=["log2FoldChange", "padj_BH"]).copy()
    d["nl"] = -np.log10(d.padj_BH.clip(lower=1e-300))
    sig = (d.padj_BH < alpha) & (d.log2FoldChange.abs() >= lfc)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    ax.scatter(d.log2FoldChange[~sig], d.nl[~sig], s=3, c="#bbbbbb")
    ax.scatter(d.log2FoldChange[sig], d.nl[sig], s=4, c="#c0392b")
    for g in d[sig].sort_values("padj_BH").head(label_n).index:
        ax.annotate(g, (d.log2FoldChange[g], d.nl[g]), fontsize=5)
    ax.axhline(-np.log10(alpha), ls="--", lw=.5, c="k"); ax.axvline(-lfc, ls="--", lw=.5, c="k"); ax.axvline(lfc, ls="--", lw=.5, c="k")
    ax.set_xlabel("log2 fold change"); ax.set_ylabel("-log10 adjusted p"); ax.set_title(title, fontsize=7)
    plt.tight_layout(); plt.savefig(f"{FIG}/{fname}.png"); plt.show()

for k, r in DE.items(): volcano(r, k, f"volcano_{k}")

## 6. Cross-dataset concordance

Core set (three or more replicates, used for discovery): HVE, HCEC, End1 L2, End1 B. Validation set (n = 2): HeLa contrasts. No expression matrices are merged. This is effect-size and direction concordance, not a formal meta-analysis.

In [ ]:
CORE = [k for k in DE if k.startswith(("GSE228774", "GSE198272", "GSE180782"))]
VALID = [k for k in DE if k.startswith("GSE158814")]
INF = CORE + VALID
print("Core:", CORE); print("Validation:", VALID)

def concordance(a, b, na, nb, alpha=0.05):
    j = a[["log2FoldChange", "padj_BH"]].join(b[["log2FoldChange", "padj_BH"]], lsuffix="_A", rsuffix="_B", how="inner").dropna()
    rho, p_rho = stats.spearmanr(j.log2FoldChange_A, j.log2FoldChange_B)
    s = j[j.padj_BH_A < alpha]
    same = int((np.sign(s.log2FoldChange_A) == np.sign(s.log2FoldChange_B)).sum()); n = len(s)
    p_bin = stats.binomtest(same, n, 0.5).pvalue if n > 0 else np.nan
    return dict(A=na, B=nb, shared_genes=len(j), spearman_rho=rho, spearman_p=p_rho, sigA=n,
                sigA_same_direction_in_B=same, frac_same=(same / n if n else np.nan), binom_p=p_bin,
                sig_in_both=int(((j.padj_BH_A < alpha) & (j.padj_BH_B < alpha)).sum()))

CONC = pd.DataFrame([concordance(DE[a], DE[b], a, b) for a, b in itertools.combinations(INF, 2)])
CONC.to_csv(f"{OUT}/cross_dataset_concordance.csv", index=False); display(CONC)

In [ ]:
def conserved(keys, alpha=0.05):
    tabs = [DE[k][["log2FoldChange", "padj_BH"]].add_suffix(f"|{k}") for k in keys]
    j = pd.concat(tabs, axis=1, join="inner").dropna()
    lf = j[[c for c in j if c.startswith("log2FoldChange")]]
    pa = j[[c for c in j if c.startswith("padj_BH")]]
    ok = (pa < alpha).all(axis=1) & ((lf > 0).all(axis=1) | (lf < 0).all(axis=1))
    out = j[ok].copy(); out["mean_log2FC"] = lf[ok].mean(axis=1)
    return out.sort_values("mean_log2FC", key=np.abs, ascending=False)

CONS = conserved(CORE) if len(CORE) >= 2 else pd.DataFrame()
CONS.to_csv(f"{OUT}/conserved_genes_core_contrasts.csv")
print("Genes significant with the same direction in ALL core contrasts:", len(CONS)); display(CONS.head(25))

rows = []
for v in VALID:
    j = CONS[["mean_log2FC"]].join(DE[v][["log2FoldChange", "padj_BH"]], how="inner").dropna()
    same = int((np.sign(j.mean_log2FC) == np.sign(j.log2FoldChange)).sum())
    rows.append(dict(validation=v, conserved_genes=len(CONS), found=len(j), same_direction=same,
                     frac_same=(same / len(j) if len(j) else np.nan), also_padj_lt_0_05=int((j.padj_BH < 0.05).sum())))
VAL = pd.DataFrame(rows); VAL.to_csv(f"{OUT}/validation_directional_check.csv", index=False); display(VAL)

## 7. Pathway analysis (preranked GSEA)

Genes are ranked by the Wald statistic. Libraries come from Enrichr through gseapy (internet needed): MSigDB Hallmark 2020, Reactome 2022, GO Biological Process 2023. Pre-specified pathways are chosen by name keyword, plus short a priori candidate lists for NLRP3, P2X7, cGAS-STING and Dectin-1 that you must review and cite. Enrichment shows transcriptional association, not functional activation.

In [ ]:
LIBS = {}
for name in ("MSigDB_Hallmark_2020", "Reactome_2022", "GO_Biological_Process_2023"):
    try:
        LIBS[name] = gp.get_library(name=name, organism="Human"); print(name, len(LIBS[name]), "sets")
    except Exception as e:
        print("Could not load", name, e)

CUSTOM = {
 "CUSTOM_NLRP3_inflammasome": ["NLRP3", "PYCARD", "CASP1", "IL1B", "IL18", "GSDMD", "NEK7", "TXNIP"],
 "CUSTOM_P2X7_axis": ["P2RX7", "PANX1", "NLRP3", "CASP1", "IL1B"],
 "CUSTOM_cGAS_STING": ["CGAS", "STING1", "TMEM173", "TBK1", "IRF3", "IFNB1", "ISG15", "CXCL10", "IFIT1", "IFIT2", "IFIT3"],
 "CUSTOM_Dectin1_CLR": ["CLEC7A", "SYK", "CARD9", "BCL10", "MALT1", "RAF1", "PLCG2"],
}
KEYWORDS = ["NF-kB", "NFkB", "Interferon", "TNF", "Interleukin-1", "NLRP3", "STING", "Dectin-1", "Inflammatory", "P2X", "Inflammasome"]
TAG = {"MSigDB_Hallmark_2020": "HALLMARK", "Reactome_2022": "REACTOME", "GO_Biological_Process_2023": "GOBP"}
GENESETS = {}
for lib, d in LIBS.items():
    for term, genes in d.items():
        if lib == "MSigDB_Hallmark_2020" or any(k.lower() in term.lower() for k in KEYWORDS):
            GENESETS[f"{TAG[lib]}|{term}"] = [g.upper() for g in genes]
GENESETS.update(CUSTOM)
print("Gene sets in analysis:", len(GENESETS))

In [ ]:
def run_gsea(r, label):
    rk = r.dropna(subset=["stat"])["stat"]
    rk = rk[~rk.index.duplicated()].sort_values(ascending=False)
    res = gp.prerank(rnk=rk, gene_sets=GENESETS, min_size=5, max_size=1000, permutation_num=1000,
                     seed=SEED, threads=2, no_plot=True, verbose=False)
    t = res.res2d.copy(); t["contrast"] = label
    for c in ("NES", "NOM p-val", "FDR q-val", "ES"): t[c] = pd.to_numeric(t[c], errors="coerce")
    t.to_csv(f"{OUT}/GSEA_{label}.csv", index=False); return t

GSEA = {}
for k, r in DE.items():
    try:
        GSEA[k] = run_gsea(r, k); print("GSEA", k, "| sets FDR<0.05:", int((GSEA[k]["FDR q-val"] < 0.05).sum()))
    except Exception as e:
        print("GSEA failed", k, e)

In [ ]:
allg = pd.concat(GSEA.values(), ignore_index=True); allg["Term"] = allg["Term"].astype(str)
pat = r"NF-kB|TNF|Interferon|Inflammatory|Interleukin-1|NLRP3|STING|Dectin|P2X|CUSTOM"
sub = allg[allg["Term"].str.contains(pat, case=False, regex=True)]
nes = sub.pivot_table(index="Term", columns="contrast", values="NES", aggfunc="first")
fdr = sub.pivot_table(index="Term", columns="contrast", values="FDR q-val", aggfunc="first")
nes.to_csv(f"{OUT}/prespecified_pathway_NES.csv"); fdr.to_csv(f"{OUT}/prespecified_pathway_FDR.csv")
top = nes.abs().max(axis=1).sort_values(ascending=False).head(40).index
fig, ax = plt.subplots(figsize=(max(7, 0.5 * nes.shape[1] + 6), 0.28 * len(top) + 2))
ann_ = fdr.loc[top].map(lambda x: "*" if pd.notna(x) and x < 0.05 else "")
sns.heatmap(nes.loc[top], cmap="RdBu_r", center=0, annot=ann_, fmt="", ax=ax, cbar_kws={"label": "NES (* FDR<0.05)"})
ax.set_xlabel(""); ax.set_ylabel(""); plt.tight_layout(); plt.savefig(f"{FIG}/pathway_NES_heatmap.png"); plt.show()

## 8. Exploratory trained-immunity comparison (GSE242947)

Gene-level concordance between each infection contrast and each ligand-versus-untrained contrast, with pathway NES in the heatmap above. Overlap does not show that epithelial cells acquire trained immunity. Time points are minutes to hours after LPS re-stimulation, so state which one you use for the comparison.

In [ ]:
mac = [k for k in DE if k.startswith("GSE242947")]
TI = pd.DataFrame([concordance(DE[a], DE[b], a, b) for a in INF for b in mac])
TI.to_csv(f"{OUT}/infection_vs_trained_concordance.csv", index=False); display(TI)

## 9. Reproducibility log and export

In [ ]:
for k, v in M.items(): v.to_csv(f"{OUT}/sample_table_{ACC[k].replace('+', '_')}.csv")
log = {"seed": SEED, "files_md5": FILE_LOG, "excluded_samples": EXCLUDE, "paired_mac": PAIRED_MAC,
       "versions": {p: im.version(p) for p in ("pydeseq2", "gseapy", "pandas", "numpy", "scipy", "statsmodels", "scikit-learn")}}
json.dump(log, open(f"{OUT}/run_log.json", "w"), indent=2)
zp = f"{BASE}/ct_reanalysis_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT):
        for f in fs: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
try:
    from google.colab import files; files.download(zp)
except Exception: print("Zip at", zp)